# Phase 4 -- Walk-Forward Parameter Tuning

Pre-registered walk-forward optimization of the Phase-2 engine's four
strategy parameters (`fvg_threshold`, `rr`, `ema_length`, `swing_lookback`),
selected in-sample per rolling fold and applied unchanged out-of-sample.

**Honest headline (read this first): `robust_improvement = FALSE` -- a
null result.** The pre-registered, falsifiable success rule required four
conditions; two of them fail (tuned stitched-OOS profit factor stays just
under 1.0, and the margin over the untuned default falls short of the
pre-registered +0.10 bar). But the *directional* finding is real and
corroborates Phase 2: walk-forward tuning very nearly erases the default's
two-year out-of-sample loss (-$907.50 vs. -$20,362.50) by trading 42% less
often and being more selective -- it just doesn't cross into robust
profitability. Full detail, all caveats, and the 4-phase program retrospective:
see `WRITEUP_PHASE4.md` in the repo root.

This notebook **loads the committed, single-shot `phase4_results.json`**
rather than re-running the walk-forward sweep (~10.6 minutes over a 144-combo
grid x 4 folds x null-control pass). The pre-registration freeze means that
JSON *is* the result -- re-running it here would defeat the point of a
single-shot, audited run (`config_hash` + `git_sha` are recorded in the file
precisely so a re-run can be told apart from the original).

## 1. Setup

In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Make relative paths (phase4_results.json, charts/...) resolve correctly
# whether Jupyter was launched from the repo root or from notebooks/.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
if not (Path.cwd() / "phase4_results.json").exists():
    raise RuntimeError(f"expected repo root, got {Path.cwd()}")
print(f"Working directory: {Path.cwd()}")

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

## 2. Load the committed results

`phase4_results.json` is the single-shot, pre-registered output of
`python run_phase4.py`. It carries its own audit trail: a SHA-256 hash of
the frozen grid/folds/floor/objective (`config_hash`) and the git commit
the run executed against (`git_sha`) -- so this notebook (and any reader)
can confirm it is narrating the actual pre-registered run, not a
retuned/rerun number.

In [ ]:
with open("phase4_results.json") as f:
    results = json.load(f)

pd.Series({
    "config_hash": results["config_hash"],
    "git_sha": results["git_sha"],
    "grid_size": results["grid_size"],
    "min_is_trades": results["min_is_trades"],
    "objective": results["objective"],
    "run_seconds": results["run_seconds"],
    "n_folds": len(results["folds"]),
})

## 3. The pre-registered verdict

The success rule (`docs/superpowers/plans/2026-07-13-phase4-parameter-tuning.md`)
was fixed **before** the run: a positive ("tuning robustly helps") verdict
requires **all four** conditions below. `robust_improvement` is the
compound AND of all four -- and it is `False`.

In [ ]:
sr = results["success_rule"]

rule_rows = [
    {
        "condition": "(a) tuned stitched-OOS PF > 1.0",
        "value": round(sr["condition_a_tuned_pf_gt_1"]["tuned_stitched_oos_pf"], 4),
        "threshold": "> 1.0",
        "passes": sr["condition_a_tuned_pf_gt_1"]["passes"],
    },
    {
        "condition": "(b) tuned - default OOS PF margin",
        "value": round(sr["condition_b_margin_gte_0_10"]["margin"], 4),
        "threshold": ">= 0.10",
        "passes": sr["condition_b_margin_gte_0_10"]["passes"],
    },
    {
        "condition": "(c) tuned beats default in >=3/4 folds",
        "value": f"{sr['condition_c_wins_3_of_4_folds']['folds_tuned_beats_default']}/"
                 f"{sr['condition_c_wins_3_of_4_folds']['total_folds']}",
        "threshold": ">= 3/4",
        "passes": sr["condition_c_wins_3_of_4_folds"]["passes"],
    },
    {
        "condition": "(d) tuned OOS PF > median-combo selection-luck null",
        "value": (f"{sr['condition_d_beats_median_combo_null']['tuned_stitched_oos_pf']:.4f} > "
                  f"{sr['condition_d_beats_median_combo_null']['median_of_fold_median_combo_oos_pf']:.4f}"),
        "threshold": "tuned > null",
        "passes": sr["condition_d_beats_median_combo_null"]["passes"],
    },
]
rule_df = pd.DataFrame(rule_rows).set_index("condition")
print(f"robust_improvement = {sr['robust_improvement']}")
rule_df

Two of four conditions fail: the tuned strategy's stitched out-of-sample
profit factor (0.9945) is still marginally under the 1.0 profitability
floor, and its margin over the untuned default (+0.0747) falls short of the
pre-registered +0.10 threshold set to rule out noise-level "improvement."
Conditions (c) and (d) *do* pass -- tuning beats the default in 3 of 4 folds
and beats the selection-luck null -- which is why this is a **null**, not a
flat **failure**: there is real directional signal, it just doesn't clear
the bar for a robust, tradeable edge.

## 4. Stitched OOS -- tuned vs. default (2024-01 .. 2025-12, never seen during tuning)

In [ ]:
stitched = results["stitched_oos"]["all_folds"]
stitched_df = pd.DataFrame(stitched).T
stitched_df.index.name = "side"
stitched_df

Tuning nearly **erases** the default's two-year loss (-$907.50 vs.
-$20,362.50 -- a >20x reduction) while trading **42% less often** (233 vs.
401 trades) and lifting profit factor from 0.92 to 0.99. It gets there
almost entirely through **selectivity**, not a different edge -- directly
corroborating Phase 2's hypothesis that the real track record's edge over
the raw default signal looks like selectivity/tuning. It just doesn't
reach breakeven.

## 5. Per-fold detail

In [ ]:
fold_rows = []
for f in results["folds"]:
    p = f["selected_params"]
    fold_rows.append({
        "fold": f["fold_index"],
        "test_window": f"{f['test_start'][:10]} .. {f['test_end'][:10]}",
        "selected (fvg/rr/ema/swing)": f"{p['fvg_threshold']}/{p['rr']}/{p['ema_length']}/{p['swing_lookback']}",
        "fallback_used": f["fallback_used"],
        "is_pf": round(f["is_pf"], 4),
        "is_n": f["is_n"],
        "oos_pf_tuned": round(f["oos_metrics"]["profit_factor"], 4),
        "oos_pf_default": round(f["oos_default_metrics"]["profit_factor"], 4),
        "tuned_beats_default": f["oos_metrics"]["profit_factor"] > f["oos_default_metrics"]["profit_factor"],
        "oos_n_tuned": f["oos_metrics"]["n_trades"],
        "oos_n_default": f["oos_default_metrics"]["n_trades"],
    })
fold_df = pd.DataFrame(fold_rows).set_index("fold")
fold_df

Fold 2 (test 2025-01..2025-06) is the one loss: the strongest in-sample
profit factor of any fold (1.194) collapsed to the weakest OOS result
(0.886, worse than the default's 1.126 on the same window) -- the largest
in-sample-to-OOS reversal in the study. `swing_lookback = 5` is the only
parameter selected consistently across all four folds; `fvg_threshold`,
`rr`, and `ema_length` scatter (see Section 8 -- read with the n=4 caveat).

## 6. The overfitting gap (in-sample PF vs. OOS PF)

In [ ]:
decay_df = pd.DataFrame(results["in_sample_to_oos_decay"]).set_index("fold_index")
decay_df.columns = ["is_pf", "oos_pf", "decay (is - oos)"]
decay_df.round(4)

In-sample PF ranges 1.01-1.21; OOS PF ranges 0.89-1.15. Three of four
folds show the expected in-sample-to-OOS decay; fold 1 went the other way
(OOS beat IS) -- a reminder that with n=4 the per-fold decay pattern isn't a
reliable signal on its own, only the aggregate is.

## 7. Is the pick special, or just lucky? -- the selection-luck null

For every fold, all 144 grid combinations' OOS profit factors were
recorded (not just the selected one). `selected_oos_percentile` is the
fraction of the 144 combos the selected pick beats OOS;
`median_combo_oos_pf` is the null a random/typical combo would have scored;
`n_combos_within_winner_ci` counts how many of the 144 combos' in-sample PF
fall inside a bootstrap confidence interval around the in-sample winner --
large values mean in-sample selection was close to a coin-flip among a
large cluster of similar combos.

In [ ]:
luck_rows = []
for f in results["folds"]:
    luck_rows.append({
        "fold": f["fold_index"],
        "selected_oos_percentile": round(f["selected_oos_percentile"], 4),
        "median_combo_oos_pf": round(f["median_combo_oos_pf"], 4),
        "selected_oos_pf": round(f["oos_metrics"]["profit_factor"], 4),
        "random_combo_oos_pf": round(f["random_combo_oos_pf"], 4),
        "n_combos_within_winner_ci (of 144)": f["n_combos_within_winner_ci"],
    })
luck_df = pd.DataFrame(luck_rows).set_index("fold")
luck_df

The picture is mixed: fold 1 (2024-07..2024-12) is genuinely special
(97th percentile of all 144 combos), but fold 0 (2024-01..2024-06) did
*worse* OOS than the median combo (30.6th percentile) -- in-sample
selection pointed the wrong way that fold. Across all folds, 72-123 of the
144 combos are statistically indistinguishable from the "winner" on
in-sample data alone. This is exactly why the pre-registered rule's
condition (d) compares against the **median**-combo null rather than just
the default -- and why passing it in aggregate should not be read as
"selection reliably finds a special combo."

## 8. Parameter stability -- read with the n=4 caveat

In [ ]:
stab = results["parameter_stability"]
stab_df = pd.DataFrame(
    {k: v for k, v in stab.items() if k != "caveat"},
    index=[f"fold {i}" for i in range(len(results["folds"]))],
).T
print("CAVEAT:", stab["caveat"])
stab_df

`swing_lookback = 5` is the only parameter chosen in all four folds.
**This is descriptive, not evidence of a real optimum**: n=4 folds has no
statistical power, and consecutive train windows overlap by 6 months (F1
and F2 share H2-2023; F2 and F3 share H1-2024; F3 and F4 share H2-2024), so
the four selected parameter-sets are not independent draws -- this
mechanically inflates any apparent agreement, `swing_lookback`'s 4-for-4
record included.

## 9. Charts

Generated by `run_phase4.py` and committed under `charts/`; loaded here
from disk to narrate alongside the tables above (matplotlib `Agg` backend
is used implicitly by non-interactive rendering, so this cell needs no
live plotting -- it only displays pre-rendered PNGs).

### Stitched OOS equity curve -- tuned vs. default

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(mpimg.imread("charts/phase4_equity_curve.png"))
ax.axis("off")
plt.show()

### Per-fold OOS profit factor -- tuned vs. default

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.imshow(mpimg.imread("charts/phase4_oos_pf_per_fold.png"))
ax.axis("off")
plt.show()

### Parameter stability table

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.imshow(mpimg.imread("charts/phase4_param_stability.png"))
ax.axis("off")
plt.show()

### In-sample vs. OOS profit factor per fold (the overfitting gap)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.imshow(mpimg.imread("charts/phase4_is_vs_oos_pf.png"))
ax.axis("off")
plt.show()

### Selection-luck null -- all 144 combos' OOS PF per fold

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
ax.imshow(mpimg.imread("charts/phase4_selection_luck_null.png"))
ax.axis("off")
plt.show()

## 10. Honest conclusion

- **`robust_improvement = False`.** The pre-registered, falsifiable success
  rule fails on 2 of 4 conditions: tuned stitched-OOS profit factor stays
  just under 1.0 (0.9945), and the margin over the default (+0.0747) falls
  short of the pre-registered +0.10 bar.
- **Directionally, tuning corroborates Phase 2.** It nearly erases the
  default's OOS loss (-$907.50 vs. -$20,362.50) by trading 42% less often
  (233 vs. 401 trades) -- selectivity helps substantially, it just doesn't
  reach profitability out-of-sample on this data.
- **The overfitting gap is real but fold-dependent.** IS PF 1.01-1.21 decays
  to OOS PF 0.89-1.15; one fold even went the other way, underscoring that
  n=4 gives no per-fold statistical power.
- **The selected picks are not consistently "special."** OOS percentile
  rank vs. the full 144-combo distribution ranges from the 31st to the 97th
  percentile across folds, and 72-123 of 144 combos are statistically
  indistinguishable from the in-sample winner in any given fold.
- **`swing_lookback = 5`** is the only stable parameter across all four
  folds -- reported descriptively, with the n=4 / overlapping-window
  caveat stated plainly, not as a discovered optimum.
- **Multiple-testing and pre-registration are disclosed, not hidden.** 144
  combos were evaluated per fold; the grid, folds, selection floor, and
  objective were frozen (`config_hash` + `git_sha`) before this single-shot
  run executed.

A rigorously measured null is a legitimate, portfolio-grade result. Full
writeup, every caveat (including the `fvg_threshold` back-adjustment
confound and the boundary-handling rules), and a closing retrospective
tying all four phases of this program together: see `WRITEUP_PHASE4.md` in
the repo root.